# Evaluation Metrics

Implement accuracy, precision, recall, F1, and confusion matrix from scratch for binary classification. Show the precision/recall tradeoff via a threshold sweep and plot a ROC curve with AUC.

## Configuration

Device, seed, and dtype come from `config.toml` via `shared.config.configure()` — never hardcoded. On Apple Silicon this runs on mps.

In [1]:
import sys
from pathlib import Path

import matplotlib

matplotlib.use("Agg")  # headless-safe under nbconvert
import matplotlib.pyplot as plt  # noqa: E402
import torch  # noqa: E402


def _find_repo_root(start: Path) -> Path:
    for p in [start, *start.parents]:
        if (p / "pyproject.toml").exists():
            return p
    return start


REPO_ROOT = _find_repo_root(Path.cwd())
sys.path.insert(0, str(REPO_ROOT))

from shared.config import configure  # noqa: E402

device = configure()
print("running on:", device)


running on: mps


In [2]:
# sklearn is available — validate against its metrics.
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    confusion_matrix as sk_confusion_matrix,
    f1_score,
    precision_score,
    r2_score,
    recall_score,
    roc_auc_score,
)
SKLEARN_AVAILABLE = True
print("sklearn available — will validate against sklearn.metrics")

sklearn available — will validate against sklearn.metrics


## Hand-computed reference values

We define a tiny 8-sample fixture whose ground-truth metrics can be verified mentally. These are used as the validation target when sklearn is not installed.

```
y_true = [1, 1, 1, 1, 0, 0, 0, 0]  (4 positives, 4 negatives)
y_pred = [1, 1, 0, 0, 0, 0, 1, 1]  (TP=2, FN=2, TN=2, FP=2)
```

From the confusion matrix:
- TP = 2, FP = 2, FN = 2, TN = 2
- Accuracy = (TP + TN) / N = 4/8 = 0.5
- Precision = TP / (TP + FP) = 2/4 = 0.5
- Recall = TP / (TP + FN) = 2/4 = 0.5
- F1 = 2 * P * R / (P + R) = 0.5

In [3]:
# Tiny fixture for deterministic hand-computed validation
y_true_tiny = torch.tensor([1, 1, 1, 1, 0, 0, 0, 0], dtype=torch.long)
y_pred_tiny = torch.tensor([1, 1, 0, 0, 0, 0, 1, 1], dtype=torch.long)
# Expected: TP=2, FP=2, FN=2, TN=2 -> acc=0.5, prec=0.5, rec=0.5, f1=0.5
REF = {"accuracy": 0.5, "precision": 0.5, "recall": 0.5, "f1": 0.5}
print("Reference values:", REF)


Reference values: {'accuracy': 0.5, 'precision': 0.5, 'recall': 0.5, 'f1': 0.5}


## From-scratch binary classification metrics

We work from the confusion matrix: a 2×2 table of true/false positives/negatives.

| | Predicted Positive | Predicted Negative |
|---|---|---|
| **Actual Positive** | TP | FN |
| **Actual Negative** | FP | TN |

- **Accuracy**: overall fraction correct. Misleading under class imbalance.
- **Precision**: among predicted positives, fraction that are truly positive. Answers: 'when we raise an alarm, how often is it real?'
- **Recall** (sensitivity, true positive rate): among actual positives, fraction detected. Answers: 'how many real cases did we catch?'
- **F1**: harmonic mean of precision and recall. Penalizes extreme tradeoffs.

In [4]:
def confusion_matrix_binary(
    y_true: torch.Tensor, y_pred: torch.Tensor
) -> tuple[int, int, int, int]:
    """Return (TP, FP, FN, TN) for binary {0, 1} tensors."""
    tp = int(((y_pred == 1) & (y_true == 1)).sum().item())
    fp = int(((y_pred == 1) & (y_true == 0)).sum().item())
    fn = int(((y_pred == 0) & (y_true == 1)).sum().item())
    tn = int(((y_pred == 0) & (y_true == 0)).sum().item())
    return tp, fp, fn, tn


def accuracy(y_true: torch.Tensor, y_pred: torch.Tensor) -> float:
    """Fraction of correctly classified samples."""
    return (y_pred == y_true).float().mean().item()


def precision_score_scratch(y_true: torch.Tensor, y_pred: torch.Tensor) -> float:
    """TP / (TP + FP). Returns 0.0 if no positive predictions."""
    tp, fp, fn, tn = confusion_matrix_binary(y_true, y_pred)
    return tp / (tp + fp) if (tp + fp) > 0 else 0.0


def recall_score_scratch(y_true: torch.Tensor, y_pred: torch.Tensor) -> float:
    """TP / (TP + FN). Returns 0.0 if no actual positives."""
    tp, fp, fn, tn = confusion_matrix_binary(y_true, y_pred)
    return tp / (tp + fn) if (tp + fn) > 0 else 0.0


def f1_score_scratch(y_true: torch.Tensor, y_pred: torch.Tensor) -> float:
    """Harmonic mean of precision and recall: 2*P*R / (P + R)."""
    p = precision_score_scratch(y_true, y_pred)
    r = recall_score_scratch(y_true, y_pred)
    return 2 * p * r / (p + r) if (p + r) > 0 else 0.0


# Validate on tiny fixture (hand-computed reference)
metrics_tiny = {
    "accuracy":  accuracy(y_true_tiny, y_pred_tiny),
    "precision": precision_score_scratch(y_true_tiny, y_pred_tiny),
    "recall":    recall_score_scratch(y_true_tiny, y_pred_tiny),
    "f1":        f1_score_scratch(y_true_tiny, y_pred_tiny),
}
print("Computed metrics on tiny fixture:")
for k, v in metrics_tiny.items():
    print(f"  {k:12s}: {v:.4f}  (expected {REF[k]:.4f})")
    assert abs(v - REF[k]) < 1e-6, f"{k} mismatch: {v} vs {REF[k]}"
print("All metrics match hand-computed reference values ✓")


Computed metrics on tiny fixture:
  accuracy    : 0.5000  (expected 0.5000)
  precision   : 0.5000  (expected 0.5000)
  recall      : 0.5000  (expected 0.5000)
  f1          : 0.5000  (expected 0.5000)
All metrics match hand-computed reference values ✓


In [5]:
# Larger synthetic dataset for sklearn comparison
torch.manual_seed(7)
N = 400
y_true_full = torch.randint(0, 2, (N,))
scores_full = torch.randn(N)  # raw scores / logits
# Bias the classifier: positive class gets higher mean score
scores_full = scores_full + 0.8 * (2 * y_true_full.float() - 1)
y_pred_full = (scores_full > 0.0).long()

acc_s  = accuracy(y_true_full, y_pred_full)
prec_s = precision_score_scratch(y_true_full, y_pred_full)
rec_s  = recall_score_scratch(y_true_full, y_pred_full)
f1_s   = f1_score_scratch(y_true_full, y_pred_full)

print(f"Scratch  | acc={acc_s:.4f}  prec={prec_s:.4f}  rec={rec_s:.4f}  f1={f1_s:.4f}")

if SKLEARN_AVAILABLE:
    yt = y_true_full.numpy()
    yp = y_pred_full.numpy()
    acc_sk  = accuracy_score(yt, yp)
    prec_sk = precision_score(yt, yp)
    rec_sk  = recall_score(yt, yp)
    f1_sk   = f1_score(yt, yp)
    print(f"sklearn  | acc={acc_sk:.4f}  prec={prec_sk:.4f}  rec={rec_sk:.4f}  f1={f1_sk:.4f}")
    assert abs(acc_s  - acc_sk)  < 1e-6, f"accuracy mismatch: {acc_s} vs {acc_sk}"
    assert abs(prec_s - prec_sk) < 1e-6, f"precision mismatch"
    assert abs(rec_s  - rec_sk)  < 1e-6, f"recall mismatch"
    assert abs(f1_s   - f1_sk)   < 1e-6, f"f1 mismatch"
    print("All metrics match sklearn ✓")
else:
    # sklearn not available: validate consistency of scratch metrics
    tp, fp, fn, tn = confusion_matrix_binary(y_true_full, y_pred_full)
    expected_prec = tp / (tp + fp)
    expected_rec  = tp / (tp + fn)
    expected_f1   = 2 * expected_prec * expected_rec / (expected_prec + expected_rec)
    assert abs(prec_s - expected_prec) < 1e-6
    assert abs(rec_s  - expected_rec)  < 1e-6
    assert abs(f1_s   - expected_f1)   < 1e-6
    print("Metrics internally consistent (sklearn unavailable) ✓")


Scratch  | acc=0.7750  prec=0.7864  rec=0.7788  f1=0.7826
sklearn  | acc=0.7750  prec=0.7864  rec=0.7788  f1=0.7826
All metrics match sklearn ✓


## Confusion matrix

A heatmap of TP, FP, FN, TN provides an at-a-glance summary of error patterns. Note that accuracy hides the distinction between FP and FN, which often have very different costs in production (e.g. a false negative in cancer screening vs a false positive).

In [6]:
def plot_confusion_matrix(y_true: torch.Tensor, y_pred: torch.Tensor, ax=None):
    """Plot a 2x2 confusion matrix heatmap."""
    tp, fp, fn, tn = confusion_matrix_binary(y_true, y_pred)
    cm = [[tn, fp], [fn, tp]]
    labels = [["TN", "FP"], ["FN", "TP"]]
    if ax is None:
        _, ax = plt.subplots()
    im = ax.imshow(cm, cmap="Blues")
    for i in range(2):
        for j in range(2):
            ax.text(j, i, f"{labels[i][j]}\n{cm[i][j]}", ha="center", va="center", fontsize=13)
    ax.set_xticks([0, 1])
    ax.set_yticks([0, 1])
    ax.set_xticklabels(["Pred 0", "Pred 1"])
    ax.set_yticklabels(["True 0", "True 1"])
    ax.set_title("Confusion Matrix")
    return ax


fig, ax = plt.subplots(figsize=(5, 4))
plot_confusion_matrix(y_true_full, y_pred_full, ax=ax)
plt.tight_layout()
plt.show()


/var/folders/gx/cg22rrrs5mx_t0gwgx3809t80000gn/T/ipykernel_68915/2167129624.py:23: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Precision / Recall tradeoff

Threshold-dependent metrics (accuracy, precision, recall, F1) depend on the decision threshold applied to classifier scores. Sweeping the threshold traces the precision-recall curve without retraining.

- A **low threshold** classifies more samples as positive: recall goes up, precision goes down.
- A **high threshold** classifies fewer samples as positive: precision goes up, recall goes down.

The threshold that maximizes F1 is often a good default operating point.

In [7]:
thresholds = torch.linspace(-3, 3, 200)

prec_curve: list[float] = []
rec_curve:  list[float] = []
f1_curve:   list[float] = []

for thr in thresholds:
    y_pred_t = (scores_full > thr).long()
    prec_curve.append(precision_score_scratch(y_true_full, y_pred_t))
    rec_curve.append(recall_score_scratch(y_true_full, y_pred_t))
    f1_curve.append(f1_score_scratch(y_true_full, y_pred_t))

best_f1_idx = int(torch.tensor(f1_curve).argmax().item())
best_thr = thresholds[best_f1_idx].item()

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# PR curve
axes[0].plot(rec_curve, prec_curve, lw=2)
axes[0].scatter(
    [rec_curve[best_f1_idx]], [prec_curve[best_f1_idx]],
    color="red", zorder=5, label=f"best F1 thr={best_thr:.2f}"
)
axes[0].set_xlabel("Recall")
axes[0].set_ylabel("Precision")
axes[0].set_title("Precision-Recall curve")
axes[0].legend()

# Metrics vs threshold
thr_np = thresholds.numpy()
axes[1].plot(thr_np, prec_curve, label="Precision")
axes[1].plot(thr_np, rec_curve, label="Recall")
axes[1].plot(thr_np, f1_curve, label="F1")
axes[1].axvline(x=best_thr, color="red", linestyle="--", label=f"best F1 @ {best_thr:.2f}")
axes[1].set_xlabel("threshold")
axes[1].set_title("Metrics vs threshold")
axes[1].legend()

plt.tight_layout()
plt.show()
print(f"Best F1: {f1_curve[best_f1_idx]:.4f} at threshold {best_thr:.3f}")


Best F1: 0.7837 at threshold 0.196


/var/folders/gx/cg22rrrs5mx_t0gwgx3809t80000gn/T/ipykernel_68915/1123105922.py:40: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## ROC curve and AUC

The Receiver Operating Characteristic (ROC) curve plots the true positive rate (recall) against the false positive rate (1 - specificity) across all thresholds. It is threshold-free and measures whether the classifier ranks positives above negatives.

$$\text{TPR} = \frac{TP}{TP + FN}, \quad \text{FPR} = \frac{FP}{FP + TN}$$

**AUC** (area under the ROC curve) has a probabilistic interpretation: it equals the probability that a randomly chosen positive sample receives a higher score than a randomly chosen negative sample. A random classifier has AUC = 0.5; a perfect classifier has AUC = 1.0.

In [8]:
def roc_curve_scratch(
    y_true: torch.Tensor, scores: torch.Tensor
) -> tuple[list[float], list[float], list[float]]:
    """Compute ROC curve by sweeping all unique score values as thresholds.

    Returns (fpr_list, tpr_list, thresholds).
    """
    thrs = scores.unique(sorted=True).flip(0)
    fpr_list: list[float] = [0.0]
    tpr_list: list[float] = [0.0]
    thr_list: list[float] = [thrs[0].item() + 1]

    total_pos = int((y_true == 1).sum().item())
    total_neg = int((y_true == 0).sum().item())

    for thr in thrs:
        y_pred_t = (scores >= thr).long()
        tp = int(((y_pred_t == 1) & (y_true == 1)).sum().item())
        fp = int(((y_pred_t == 1) & (y_true == 0)).sum().item())
        fpr_list.append(fp / total_neg if total_neg > 0 else 0.0)
        tpr_list.append(tp / total_pos if total_pos > 0 else 0.0)
        thr_list.append(thr.item())

    fpr_list.append(1.0)
    tpr_list.append(1.0)
    return fpr_list, tpr_list, thr_list


def auc_trapezoidal(fpr: list[float], tpr: list[float]) -> float:
    """Area under ROC via trapezoidal rule."""
    total = 0.0
    for i in range(1, len(fpr)):
        dx = fpr[i] - fpr[i - 1]
        total += dx * (tpr[i] + tpr[i - 1]) / 2
    return abs(total)


fpr, tpr, thr_list = roc_curve_scratch(y_true_full, scores_full)
auc_scratch = auc_trapezoidal(fpr, tpr)
print(f"AUC (scratch trapezoidal): {auc_scratch:.4f}")

# --- Tight sklearn validation (sklearn is now installed) ---
auc_sk = roc_auc_score(y_true_full.numpy(), scores_full.numpy())
print(f"AUC (sklearn):             {auc_sk:.4f}")
assert abs(auc_scratch - auc_sk) < 1e-3, f"AUC mismatch: {auc_scratch:.6f} vs {auc_sk:.6f}"
print("AUC matches sklearn within 1e-3 ✓")

# --- Extra check: rank-based (Mann-Whitney U) AUC ---
# P(score_pos > score_neg) computed from ranks.
pos_scores = scores_full[y_true_full == 1].numpy()
neg_scores = scores_full[y_true_full == 0].numpy()
n_pos, n_neg = len(pos_scores), len(neg_scores)
u_stat = sum(
    1.0 if p > n else (0.5 if p == n else 0.0)
    for p in pos_scores for n in neg_scores
)
auc_mwu = u_stat / (n_pos * n_neg)
print(f"AUC (Mann-Whitney U):      {auc_mwu:.4f}")
assert abs(auc_mwu - auc_sk) < 1e-4, f"MWU AUC mismatch vs sklearn: {auc_mwu} vs {auc_sk}"
print("Mann-Whitney U AUC matches sklearn ✓")

AUC (scratch trapezoidal): 0.8628
AUC (sklearn):             0.8628
AUC matches sklearn within 1e-3 ✓
AUC (Mann-Whitney U):      0.8628
Mann-Whitney U AUC matches sklearn ✓


In [9]:
fig, ax = plt.subplots(figsize=(6, 5))
ax.plot(fpr, tpr, lw=2, label=f"ROC curve (AUC = {auc_scratch:.3f})")
ax.plot([0, 1], [0, 1], linestyle="--", color="gray", label="random classifier")
ax.set_xlabel("False Positive Rate")
ax.set_ylabel("True Positive Rate (Recall)")
ax.set_title("ROC Curve")
ax.legend()
plt.tight_layout()
plt.show()


/var/folders/gx/cg22rrrs5mx_t0gwgx3809t80000gn/T/ipykernel_68915/3307460700.py:9: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## When accuracy is misleading

With 95% negatives and 5% positives, a classifier that predicts 'negative' for everything achieves 95% accuracy but catches zero positives. Precision, recall, and AUC expose this failure.

In [10]:
torch.manual_seed(99)
N_imb = 1000
y_imb = torch.cat([torch.ones(50, dtype=torch.long), torch.zeros(950, dtype=torch.long)])
y_pred_all_neg = torch.zeros(N_imb, dtype=torch.long)  # trivial all-negative classifier

acc_imb  = accuracy(y_imb, y_pred_all_neg)
prec_imb = precision_score_scratch(y_imb, y_pred_all_neg)
rec_imb  = recall_score_scratch(y_imb, y_pred_all_neg)
f1_imb   = f1_score_scratch(y_imb, y_pred_all_neg)

print(f"All-negative classifier on 5% positive data:")
print(f"  Accuracy:  {acc_imb:.3f}  ← looks great!")
print(f"  Precision: {prec_imb:.3f}")
print(f"  Recall:    {rec_imb:.3f}  ← catches zero positives")
print(f"  F1:        {f1_imb:.3f}")

assert acc_imb > 0.9, "Expected high accuracy for all-negative on imbalanced data"
assert rec_imb == 0.0, "All-negative classifier should have zero recall"
assert f1_imb == 0.0, "All-negative classifier should have zero F1"
print("Accuracy misleads under class imbalance; recall and F1 expose the failure ✓")


All-negative classifier on 5% positive data:
  Accuracy:  0.950  ← looks great!
  Precision: 0.000
  Recall:    0.000  ← catches zero positives
  F1:        0.000
Accuracy misleads under class imbalance; recall and F1 expose the failure ✓


## Regression Metrics

Regression metrics quantify prediction error in the target's native units (MAE, RMSE) or relative to a trivial baseline (R²).

| Metric | Formula | Measures |
|---|---|---|
| MAE | `(1/n) Σ|ŷ - y|` | Average absolute error; easy to interpret in target units |
| MSE | `(1/n) Σ(ŷ - y)²` | Mean squared error; penalises large residuals quadratically |
| RMSE | `√MSE` | Same scale as the target; directly comparable to MAE |
| R² | `1 - SS_res / SS_tot` | Fraction of variance explained; 1 = perfect, 0 = mean baseline |

R² can be negative when the model is worse than predicting the mean.

In [11]:
import torch.nn.functional as F
# Synthetic regression data (reuse y_reg / y_hat from the top of the notebook,
# but rebuild here so this section is self-contained)
torch.manual_seed(42)
N_reg = 200
y_reg_val  = torch.randn(N_reg, device=device)
y_hat_val  = y_reg_val + 0.4 * torch.randn(N_reg, device=device)


def mae_metric(y_hat: torch.Tensor, y: torch.Tensor) -> torch.Tensor:
    """Mean absolute error: (1/N) Σ |ŷ_i - y_i|."""
    return (y_hat - y).abs().mean()


def mse_metric(y_hat: torch.Tensor, y: torch.Tensor) -> torch.Tensor:
    """Mean squared error: (1/N) Σ (ŷ_i - y_i)²."""
    return ((y_hat - y) ** 2).mean()


def rmse_metric(y_hat: torch.Tensor, y: torch.Tensor) -> torch.Tensor:
    """Root mean squared error: √MSE."""
    return mse_metric(y_hat, y).sqrt()


def r2_metric(y_hat: torch.Tensor, y: torch.Tensor) -> torch.Tensor:
    """Coefficient of determination: 1 - SS_res / SS_tot."""
    ss_res = ((y - y_hat) ** 2).sum()
    ss_tot = ((y - y.mean()) ** 2).sum()
    return 1.0 - ss_res / ss_tot


# --- Compute ---
mae_v  = mae_metric(y_hat_val, y_reg_val)
mse_v  = mse_metric(y_hat_val, y_reg_val)
rmse_v = rmse_metric(y_hat_val, y_reg_val)
r2_v   = r2_metric(y_hat_val, y_reg_val)

print(f"MAE  (scratch): {mae_v.item():.6f}")
print(f"MSE  (scratch): {mse_v.item():.6f}")
print(f"RMSE (scratch): {rmse_v.item():.6f}")
print(f"R²   (scratch): {r2_v.item():.6f}")

# --- Validate MAE vs F.l1_loss ---
ref_mae = F.l1_loss(y_hat_val, y_reg_val)
assert torch.allclose(mae_v, ref_mae, atol=1e-6), f"MAE mismatch: {mae_v} vs {ref_mae}"
print("MAE matches F.l1_loss ✓")

# --- Validate MSE vs F.mse_loss ---
ref_mse = F.mse_loss(y_hat_val, y_reg_val)
assert torch.allclose(mse_v, ref_mse, atol=1e-6), f"MSE mismatch: {mse_v} vs {ref_mse}"
print("MSE matches F.mse_loss ✓")

# --- Validate RMSE vs sqrt(F.mse_loss) ---
ref_rmse = F.mse_loss(y_hat_val, y_reg_val).sqrt()
assert torch.allclose(rmse_v, ref_rmse, atol=1e-6), f"RMSE mismatch: {rmse_v} vs {ref_rmse}"
print("RMSE matches sqrt(F.mse_loss) ✓")

# --- Validate R² vs sklearn ---
r2_sk = r2_score(y_reg_val.cpu().numpy(), y_hat_val.cpu().numpy())
assert abs(r2_v.item() - r2_sk) < 1e-6, f"R² mismatch: {r2_v.item()} vs {r2_sk}"
print(f"R² matches sklearn.metrics.r2_score ({r2_sk:.6f}) ✓")

MAE  (scratch): 0.314541
MSE  (scratch): 0.162404
RMSE (scratch): 0.402994
R²   (scratch): 0.828856


MAE matches F.l1_loss ✓


MSE matches F.mse_loss ✓
RMSE matches sqrt(F.mse_loss) ✓
R² matches sklearn.metrics.r2_score (0.828856) ✓


## PR-AUC (Average Precision)

PR-AUC summarises the precision-recall curve and is more informative than ROC-AUC when the positive class is rare, because it focuses on positive predictions and ignores the large pool of true negatives.

Two common estimators:
- **Trapezoidal PR-AUC**: area under the PR curve via the trapezoidal rule over the threshold sweep.
- **Average Precision (AP)**: `Σ_k (R_k - R_{k-1}) · P_k`, which is a step-interpolated version of the same integral. AP and trapezoidal PR-AUC use different interpolation so they will differ slightly.

We implement trapezoidal PR-AUC from scratch and validate it against a trapezoidal reference computed from the same (precision, recall) points. We also report sklearn's AP value for comparison.

In [12]:
def pr_auc_trapezoidal(
    y_true: torch.Tensor, scores: torch.Tensor
) -> tuple[float, list[float], list[float]]:
    """Compute precision-recall AUC via trapezoidal rule over a threshold sweep.

    Returns (pr_auc, precision_list, recall_list) sorted by increasing recall.
    """
    # Sweep thresholds from high to low (high threshold -> low recall, high precision)
    thrs = scores.unique(sorted=True).flip(0)
    total_pos = int((y_true == 1).sum().item())

    prec_pts: list[float] = []
    rec_pts:  list[float] = []

    for thr in thrs:
        y_pred_t = (scores >= thr).long()
        tp = int(((y_pred_t == 1) & (y_true == 1)).sum().item())
        fp = int(((y_pred_t == 1) & (y_true == 0)).sum().item())
        prec_pts.append(tp / (tp + fp) if (tp + fp) > 0 else 1.0)
        rec_pts.append(tp / total_pos  if total_pos > 0 else 0.0)

    # Add boundary points: (recall=0, precision=1) and (recall=1, precision=last)
    # Sort by recall ascending for trapezoidal integration
    pairs = sorted(zip(rec_pts, prec_pts))
    rec_sorted  = [0.0] + [p[0] for p in pairs]
    prec_sorted = [1.0] + [p[1] for p in pairs]

    # Trapezoidal rule
    area = 0.0
    for i in range(1, len(rec_sorted)):
        dr = rec_sorted[i] - rec_sorted[i - 1]
        area += dr * (prec_sorted[i] + prec_sorted[i - 1]) / 2.0

    return abs(area), prec_sorted, rec_sorted


pr_auc_scratch, prec_pts, rec_pts = pr_auc_trapezoidal(y_true_full, scores_full)
print(f"PR-AUC (scratch trapezoidal): {pr_auc_scratch:.6f}")

# --- Validate against sklearn.metrics.auc (independent trapezoidal reference) ---
# sklearn.metrics.auc(x, y) integrates y over x via the trapezoidal rule, so it
# must match our trapezoidal PR-AUC to floating-point precision.
from sklearn.metrics import auc as sklearn_auc
trap_ref = sklearn_auc(rec_pts, prec_pts)
assert abs(pr_auc_scratch - trap_ref) < 1e-6, (
    f"PR-AUC mismatch: {pr_auc_scratch} vs {trap_ref}"
)
print("PR-AUC matches sklearn.metrics.auc (independent trapezoidal reference) ✓")

# --- Compare to sklearn average_precision_score (step-interpolation, so values differ) ---
ap_sk = average_precision_score(y_true_full.numpy(), scores_full.numpy())
print(f"sklearn average_precision_score (step AP): {ap_sk:.6f}")
print(
    f"  Note: trapezoidal PR-AUC ({pr_auc_scratch:.4f}) and sklearn AP ({ap_sk:.4f}) use "
    "different interpolation; a small difference is expected."
)

# Tight assertion: our trapezoidal value is within reasonable range of AP.
# Both should be clearly > 0.5 for this well-separated classifier, and they
# should agree within ~0.05 (the interpolation difference).
assert abs(pr_auc_scratch - ap_sk) < 0.05, (
    f"PR-AUC and AP diverge unexpectedly: {pr_auc_scratch:.4f} vs {ap_sk:.4f}"
)
print("Trapezoidal PR-AUC and sklearn AP agree within interpolation tolerance ✓")

# --- Plot PR curve ---
fig_pr, ax_pr = plt.subplots(figsize=(6, 5))
ax_pr.plot(rec_pts, prec_pts, lw=2, label=f"PR curve (trap AUC = {pr_auc_scratch:.3f})")
ax_pr.axhline(
    y=y_true_full.float().mean().item(), linestyle="--", color="gray",
    label=f"random (prevalence = {y_true_full.float().mean().item():.2f})"
)
ax_pr.set_xlabel("Recall")
ax_pr.set_ylabel("Precision")
ax_pr.set_title("Precision-Recall Curve")
ax_pr.legend()
plt.tight_layout()
plt.show()
print(f"PR-AUC from scratch: {pr_auc_scratch:.4f}  |  sklearn AP: {ap_sk:.4f}")

PR-AUC (scratch trapezoidal): 0.887930
PR-AUC matches sklearn.metrics.auc (independent trapezoidal reference) ✓
sklearn average_precision_score (step AP): 0.888298
  Note: trapezoidal PR-AUC (0.8879) and sklearn AP (0.8883) use different interpolation; a small difference is expected.
Trapezoidal PR-AUC and sklearn AP agree within interpolation tolerance ✓
PR-AUC from scratch: 0.8879  |  sklearn AP: 0.8883


/var/folders/gx/cg22rrrs5mx_t0gwgx3809t80000gn/T/ipykernel_68915/811765199.py:78: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Takeaways

- **Accuracy** answers 'how often are we right?' but is misleading when classes are imbalanced.
- **Precision** = TP / (TP + FP): when we raise an alarm, how often is it real? Use when false positives are costly.
- **Recall** (TPR, sensitivity) = TP / (TP + FN): how many real cases did we catch? Use when false negatives are costly.
- **F1** is the harmonic mean of precision and recall. It punishes systems that achieve one at the expense of the other.
- **ROC-AUC** is threshold-free: it measures whether the classifier ranks positives above negatives. AUC = 0.5 is random; AUC = 1.0 is perfect. It is less informative when the positive class is rare — in that case, PR-AUC focuses on positive predictions.
- Threshold sweeping shows the precision-recall tradeoff without retraining. The optimal threshold depends on the relative cost of false positives vs false negatives.
- Metrics are not losses: they measure task success on held-out data. The training loss (e.g. cross-entropy) is a differentiable proxy for the metric you actually care about.
- **MAE, MSE, RMSE** measure regression error in interpretable units. MAE is robust to outliers (bounded subgradient); RMSE penalises large errors quadratically. R² measures the fraction of variance explained; 0 = predicting the mean, 1 = perfect fit.
- **PR-AUC** summarises the precision-recall curve. It is more informative than ROC-AUC when the positive class is rare. Trapezoidal PR-AUC and sklearn's average_precision_score (step interpolation) differ slightly; both are valid.